<img src="https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# Portrait Segmentation (Portrait128) — Akida 2 Benchmark

Run Time: TBD

This notebook walks through evaluating and benchmarking a model on the **Portrait128** dataset, targeting Akida 2 hardware.

For details on the dataset and preparation of the model, see the neighbouring README.md and `portrait128_notebook_training.ipynb`.

> **Note:** the hardware benchmark needs a physical Akida 2 FPGA device. Every hardware step is guarded, so the notebook runs end to end without a board: the latency cells are skipped, and sparsity is still computed on the software backend. It will not produce hardware numbers on Colab.

The Akida 2 reference platform is an FPGA running at 25 MHz. Alongside the measured latency, the notebook reports a projected latency at the AKD2500 target clock. Power is not measured yet, so the benchmark is latency-only.

## Setup

In [ ]:
import os
import numpy as np
import akida

from brainchip_utils.hardware_utils import AKIDA_CLOCKS_HZ

DATA_PATH = './data/portrait128'
INPUT_SHAPE = (128, 128, 3)

# Pretrained models committed to the repo. If you trained your own with the
# training notebook or portrait128_train.sh, set MODELS_DIR = './models/'.
MODELS_DIR = './pretrained_models/'

# Which converted model to benchmark. Options produced by the training pipeline:
#   akida_unet_portrait128_i8_w8_a8.fbz     (8-bit)
#   akida_unet_portrait128_i8_w4_a4_qat.fbz (4-bit QAT)
MODEL_FILENAME = 'akida_unet_portrait128_i8_w4_a4_qat.fbz'

# Clocks (see portrait128_benchmark.py and brainchip_utils.hardware_utils.AKIDA_CLOCKS_HZ).
MEASURED_CLOCK = AKIDA_CLOCKS_HZ['AKIDA2_FPGA']  # 25 MHz FPGA
PROJECTED_CLOCK = AKIDA_CLOCKS_HZ['AKD2500']     # 1 GHz target (pre-production)

## Model

The notebook loads a converted Akida model (`.fbz`) from `pretrained_models/`, for example
`pretrained_models/akida_unet_portrait128_i8_w4_a4_qat.fbz`. Pretrained models are committed
through `git-lfs`; see the [Trained models](../../../README.md#trained-models) section of the
top-level README for set-up.

📝 Planned: the pretrained models for this example are added once the training pipeline has
been run. Until then, train your own with `portrait128_notebook_training.ipynb` or
`portrait128_train.sh` and point `MODELS_DIR` at `./models/`.

The model takes a uint8 RGB image of 128 × 128 and returns one logit per pixel, as a
`(128, 128, 1)` map. Its first layer, `InputConv2D`, runs on Akida's dedicated input
processor, which takes the uint8 image directly.

In [ ]:
ak_model = akida.Model(os.path.join(MODELS_DIR, MODEL_FILENAME))
ak_model.summary()

## Device

`get_akida_device` returns `None` when no compatible hardware is present, in which case the latency benchmark below is skipped.

In [ ]:
from brainchip_utils.hardware_utils import get_akida_device

device = get_akida_device(target_version=ak_model.ip_version)
if device is None:
    print('No compatible Akida hardware device found — benchmark will be skipped.')
else:
    print('Akida device found:', device)

## Samples

Akida's neural processors only do work for non-zero activations, so latency depends on the input. Random or synthetic images drive unrepresentative activity and give misleading timings: a random image has no smooth regions, and its activations are much less sparse than a real portrait's. So the benchmark runs on real images from the training split.

In [ ]:
from portrait128_data import get_samples

NUM_SAMPLES = 100
samples = get_samples(DATA_PATH, INPUT_SHAPE, num_samples=NUM_SAMPLES)

## Latency benchmark

The model is benchmarked in each of the three mapping modes, which trade the number of neural processors (NPs) used against the number of passes:

* **Minimal** maps the model onto the fewest NPs that can hold it.
* **AllNps** spreads it over as many NPs as possible, while keeping the minimum number of passes.
* **HwPr** (hardware partial reconfiguration) splits the model into several passes so each layer can use more NPs, reloading weights between passes on every inference.

Which mode is fastest depends on the model, so all three are measured.

The FPGA runs at 25 MHz, far below production silicon. The number of clock cycles per inference is fixed for a given model and mapping, whatever the clock, so the latency at the AKD2500 target clock is an exact rescale: `projected_ms = mean_inf_clk / PROJECTED_CLOCK * 1000`.

In [ ]:
from brainchip_utils.hardware_utils import full_model_benchmark, get_mapping_stats

if device is not None:
    for mm in ['Minimal', 'AllNps', 'HwPr']:
        map_mode = getattr(akida.MapMode, mm)
        res = full_model_benchmark(ak_model, device, samples,
                                   map_mode=map_mode, clock_freq=MEASURED_CLOCK)
        projected_ms = res['mean_inf_clk'] / PROJECTED_CLOCK * 1000
        ak_model.map(device, mode=map_mode)
        num_nps, num_passes, num_sequences = get_mapping_stats(ak_model)
        print(f'[{mm}] NPs={num_nps} passes={num_passes} '
              f'latency@25MHz={res["mean_clk_ms"]:.3f} ms '
              f'projected@1000MHz={projected_ms:.3f} ms')
else:
    print('Hardware not available — skipping latency benchmark.')

## Per-layer benchmark & sparsity

The per-layer benchmark (Minimal mapping) shows where the time goes, and the sparsity printout shows why. A layer's cost grows with the number of non-zero activations it receives, so on Akida, per-layer latency and input sparsity are linked: a layer with dense input and a large feature map is slow.

In this model, look at the last decoder blocks. They run at 64 × 64 and 128 × 128, so they receive far more activations than the encoder layers at 8 × 8 or 4 × 4. Their sparsity therefore matters most.

Sparsity doesn't need hardware, so it is also computed on the software backend when no device is present.

In [ ]:
from brainchip_utils.hardware_utils import per_layer_benchmark
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity

if device is not None:
    ak_model.map(device, mode=akida.MapMode.Minimal, hw_only=True)
    sparsity_dict = compute_sparsity(ak_model, samples=samples)
    pretty_print_sparsity(sparsity_dict)
    per_layer_results = per_layer_benchmark(ak_model, device, samples,
                                            repeats=NUM_SAMPLES, clock_freq=MEASURED_CLOCK)
    print('Per-layer benchmark complete.')
else:
    # Sparsity can still be computed on the software backend without hardware.
    sparsity_dict = compute_sparsity(ak_model, samples=samples)
    pretty_print_sparsity(sparsity_dict)
    print('Hardware not available — skipped latency; sparsity computed on software backend.')